# Timbão Batucada — Séparation en stems (Drumsep)

Sépare un morceau en **familles** de percussions :
- **bombo** → groupe **surdos** (grave)
- **redoblante** → **caixa** (caisse claire)
- **toms** → **repinike**
- platillos → (cymbales, ignoré ici)

⚠️ **Limite physique** : Drumsep sépare des familles, **pas** les 3 surdos individuels (marcação / doubles / coupes). Pour ça il faut un enregistrement multipiste.

**Mode d'emploi :** `Exécution → Modifier le type d'exécution → GPU`, puis lance les cellules dans l'ordre. À la fin tu télécharges un zip de stems que tu m'envoies (ou que tu déposes dans le projet).

In [ ]:
# 1) Installation (demucs + modèle Drumsep)
!pip -q install demucs requests
!git clone -q https://github.com/inagoy/drumsep.git
print('Drumsep prêt')

In [ ]:
# 2) Choisis le morceau et récupère son audio depuis la bibliothèque
APP_URL = 'https://timbao-batucada-music.vercel.app'
TITLE   = 'Bombareg'   # <-- change le titre ici

import requests, re, os
tracks = requests.get(APP_URL + '/api/tracks', timeout=60).json()
t = next((x for x in tracks if x['title'].lower() == TITLE.lower()), None)
assert t, f'Morceau introuvable: {TITLE}'
ext = '.mp3' if t['audioUrl'].lower().endswith('.mp3') else '.m4a'
open('input'+ext, 'wb').write(requests.get(t['audioUrl'], timeout=120).content)
print('Téléchargé :', t['title'], '·', round(t['durationSec']), 's →', 'input'+ext)

In [ ]:
# 3) Séparation (première exécution : télécharge le modèle, peut prendre quelques minutes)
import glob
src = glob.glob('input.*')[0]
!cd drumsep && bash drumsep "../{src}" ../stems_out
print('\nFichiers produits :')
!find stems_out -name '*.wav'

In [ ]:
# 4) Renomme par instrument + zip à télécharger
import glob, shutil, os
MAP = {'bombo':'surdos', 'redoblante':'caixa', 'toms':'repinike', 'platillos':'platillos'}
os.makedirs('stems_'+TITLE, exist_ok=True)
for w in glob.glob('stems_out/**/*.wav', recursive=True):
    base = os.path.splitext(os.path.basename(w))[0].lower()
    name = MAP.get(base, base)
    shutil.copy(w, f'stems_{TITLE}/{name}.wav')
print('Stems :', os.listdir('stems_'+TITLE))
shutil.make_archive('stems_'+TITLE, 'zip', 'stems_'+TITLE)
from google.colab import files
files.download(f'stems_{TITLE}.zip')

## Ensuite
Envoie-moi le zip `stems_<TITRE>.zip` (ou dépose-le dans `que-calor-samba/music tracks/<TITRE>/stems/`).

Je m'occupe du reste : analyse propre de chaque stem (onsets par piste, calés sur la grille du morceau), upload des stems, et mise à jour du morceau pour que **soloer un instrument joue son vrai son isolé** avec une partition juste (au niveau famille).

Commence par **un** morceau (Bombareg) pour valider la chaîne, puis on enchaîne les autres.